### Notebook to visualize the domain boundaries output

In [2]:
import os
import pandas as pd
import numpy as np


# Path to the _domains result file
domains_path = "/scratch/ymeng/DPAM/input/test64Threads_domains"

# Read the domains file as a dataframe
df_domains = pd.read_csv(domains_path, sep='\t')

In [ ]:
# Select the protein to visualize
unique_proteins = df_domains['Protein'].unique()
protein_id = unique_proteins[6]
df_protein = df_domains[df_domains['Protein'] == protein_id]

df_protein

,Protein,Domain,Range,ECOD_num,ECOD_key,T-group,DPAM_prob,HH_prob,DALI_zscore,Hit_cov,Tgroup_cov,Judge,Hcount,Scount
28,P17948,nD1,26-135,1678261,e4wrlA2,11.1.1,1.000,0.997,17.6,0.486,1.019,good_domain,1,7
29,P17948,nD2,136-225,1678261,e4wrlA2,11.1.1,1.000,0.997,17.6,0.432,0.833,good_domain,1,8
30,P17948,nD3,226-330,1954714,e5t89X3,11.1.1,1.000,1.000,17.3,0.944,0.972,good_domain,1,7
31,P17948,nD4,331-425,1954718,e5t89Y2,11.1.1,1.000,1.000,19.8,0.418,0.880,good_domain,1,7
32,P17948,nD5,426-555,1954718,e5t89Y2,11.1.1,1.000,1.000,19.8,0.514,1.204,good_domain,0,8
33,P17948,nD6,556-655,1954716,e5t89X5,11.1.1,1.000,1.000,10.5,0.739,0.926,good_domain,2,7
34,P17948,nD7,656-785,379873,e3jxaB4,11.1.1,0.999,0.999,15.8,1.000,1.204,good_domain,3,8
35,P17948,nD8,"786-950,986-1170",1166011,e4aseA2,206.1.1,0.983,1.000,35.7,0.917,1.163,good_domain,11,8


In [4]:
str.replace(df_protein.iloc[0]['Range'], ",", " resi ")

'26-135'

In [5]:
# Get the PDB file path
model_dir = str.replace(os.path.abspath(domains_path), "_domains", "")
model_path = os.path.join(model_dir, f"{protein_id}.pdb")
print(model_path)

/scratch/ymeng/DPAM/input/test64Threads/P17948.pdb


Problematic entries:
- P35499

In [7]:
import py3Dmol

def visualize_protein_domains(df_protein):
    """
    Visualizes protein domains from a DataFrame row subset using py3Dmol.

    Args:
        df_protein (pd.DataFrame): Subset of domains DataFrame for a specific protein.
                                  Must have columns 'Domain', 'Range', 'Judge', and 'Protein'.

    Returns:
        py3Dmol.view: The rendered 3Dmol.js view object.
    """
    # Define distinct colors for domains (expand as needed)
    domain_colors = [
        'red', 'green', 'blue', 'orange', 'magenta', 'cyan', 'yellow', 'purple', 'brown', 'lime'
    ]

    def parse_ranges(range_string):
        """Return list of (start, end) tuples (1-based inclusive) from a 'Range' string like '6-35,41-410'."""
        res = []
        for part in range_string.split(','):
            start, end = [int(x) for x in part.split('-')]
            res.append((start, end))
        return res

    # Determine model path from DataFrame
    protein_id = df_protein.iloc[0]['Protein']
    # Assumes domains_path was defined in the outer scope, as in the notebook above
    # Get the model_dir and model_path
    model_dir = str.replace(os.path.abspath(domains_path), "_domains", "")
    model_path = os.path.join(model_dir, f"{protein_id}.pdb")

    view = py3Dmol.view(width=500, height=500)
    view.addModel(open(model_path).read(), 'pdb')

    # Set the default structure color to lightgrey
    view.setStyle({'cartoon': {'color': 'lightgrey'}})

    # Color each domain in df_protein a unique color
    for idx, (domain_name, range_str, judge) in enumerate(zip(df_protein['Domain'], df_protein['Range'], df_protein['Judge'])):
        color = domain_colors[idx % len(domain_colors)]
        domain_ranges = parse_ranges(range_str)
        for start, end in domain_ranges:
            # Color the residues for the domain
            selection = {'resi': list(range(start, end + 1))}
            view.setStyle(selection, {'cartoon': {'color': color}})
    
    view.zoomTo()
    return view

# example usage
visualize_protein_domains(df_protein)


3Dmol.js failed to load for some reason. Please check your browser console for error messages.